# 🔗 PNG to Label Mapping - Google Colab

**Objectif:** Mapper les PNG aux labels à partir de:
- `png_files_list.csv` (liste des PNG avec DICOM IDs)
- `dataset_labeled_enriched.csv` (labels)

**Output:** `png_label_mapping.csv` avec colonnes: png_filename | dicom_id | label

## 📦 Section 1: Installer les dépendances et monter Google Drive

In [ ]:
import subprocess
import sys

# Installer les packages (si nécessaire)
packages = ['pandas', 'numpy']
for package in packages:
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import pandas as pd
import numpy as np
import os
from pathlib import Path

print("✅ Packages importés avec succès!")

In [ ]:
# Monter Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive monté!")

## 📂 Section 2: Configurer les chemins et charger les CSV

In [ ]:
# Configuration des chemins
DATASET_FOLDER = '/content/drive/MyDrive/dataset_analysis_colab'  # 📝 Adapter si nécessaire

PNG_FILES_LIST = os.path.join(DATASET_FOLDER, 'png_files_list.csv')
DATASET_CSV = os.path.join(DATASET_FOLDER, 'dataset_labeled_enriched.csv')
OUTPUT_CSV = os.path.join(DATASET_FOLDER, 'png_label_mapping.csv')

print(f"📁 Dossier dataset: {DATASET_FOLDER}")
print(f"📄 PNG files list: {PNG_FILES_LIST}")
print(f"📄 Dataset CSV: {DATASET_CSV}")
print(f"📄 Output (mapping): {OUTPUT_CSV}")

# Vérifier que les fichiers existent
if not os.path.exists(PNG_FILES_LIST):
    print(f"❌ Fichier NOT FOUND: {PNG_FILES_LIST}")
if not os.path.exists(DATASET_CSV):
    print(f"❌ Fichier NOT FOUND: {DATASET_CSV}")

In [ ]:
# Charger les données
print(f"\n{'='*70}")
print("📊 CHARGEMENT DES DONNÉES")
print(f"{'='*70}")

try:
    df_png_list = pd.read_csv(PNG_FILES_LIST)
    print(f"\n✅ PNG Files List: {len(df_png_list)} images")
    print(f"   Colonnes: {list(df_png_list.columns)}")
    print(f"\n   Premiers 3 fichiers:")
    print(df_png_list.head(3).to_string())
except Exception as e:
    print(f"❌ Erreur: {e}")
    raise

In [ ]:
try:
    df_dataset = pd.read_csv(DATASET_CSV)
    print(f"\n✅ Dataset: {len(df_dataset)} entrées")
    print(f"   Colonnes: {list(df_dataset.columns)}")
    print(f"\n   Premiers 3 enregistrements:")
    print(df_dataset.head(3).to_string())
except Exception as e:
    print(f"❌ Erreur: {e}")
    raise

## 🏷️ Section 3: Identifier la colonne label

In [ ]:
print(f"\n{'='*70}")
print("🏷️  IDENTIFICATION DE LA COLONNE LABEL")
print(f"{'='*70}")

label_col = None
candidates = ['finding', 'label', 'diagnosis', 'class']

for col in candidates:
    if col in df_dataset.columns:
        label_col = col
        print(f"✅ Colonne trouvée: '{label_col}'")
        break

if label_col is None:
    # Si aucune colonne standard, afficher les colonnes disponibles
    print(f"❌ Aucune colonne standard trouvée")
    print(f"\n   Colonnes disponibles:")
    for i, col in enumerate(df_dataset.columns, 1):
        print(f"      {i}. {col}")
    
    # Par défaut, utiliser la dernière
    label_col = df_dataset.columns[-1]
    print(f"\n   ℹ️  Utilisant par défaut: '{label_col}'")

print(f"\n   Exemples de labels:")
print(df_dataset[label_col].value_counts().head(10).to_string())

## 🔗 Section 4: Extraire les DICOM IDs et faire le mapping

In [ ]:
print(f"\n{'='*70}")
print("🔗 CRÉATION DU MAPPING PNG → DICOM ID → LABEL")
print(f"{'='*70}")

mapping_data = []
matched_count = 0
unmatched_files = []

print(f"\nTraitement de {len(df_png_list)} fichiers PNG...\n")

for idx, row in df_png_list.iterrows():
    png_filename = row['filename']
    
    try:
        # Extraire le DICOM ID du nom PNG
        # Pattern: "2035_IM-0680-1001_multiwindow.png"
        # On cherche le "IM-" et on extrait de là
        
        # Supprimer l'extension et le suffixe _multiwindow
        base_name = png_filename.replace('_multiwindow.png', '').replace('.png', '')
        
        # Trouver la position de "IM-"
        im_pos = base_name.find('IM-')
        if im_pos == -1:
            unmatched_files.append((png_filename, "Pas de DICOM ID trouvé"))
            continue
        
        dicom_id = base_name[im_pos:]  # "IM-0680-1001"
        
        # Chercher dans le dataset
        matching_rows = df_dataset[
            df_dataset.astype(str).apply(
                lambda x: x.str.contains(dicom_id, na=False, case=False)
            ).any(axis=1)
        ]
        
        if len(matching_rows) > 0:
            label = matching_rows[label_col].iloc[0]
            
            mapping_data.append({
                'png_filename': png_filename,
                'dicom_id': dicom_id,
                'label': label
            })
            matched_count += 1
            
            # Afficher les 5 premiers pour vérification
            if matched_count <= 5:
                print(f"   ✅ {png_filename} → {dicom_id} → {label}")
        else:
            unmatched_files.append((png_filename, f"DICOM ID '{dicom_id}' not in dataset"))
    
    except Exception as e:
        unmatched_files.append((png_filename, str(e)))

print(f"\n... (traitement des autres fichiers)\n")

## 📊 Section 5: Analyse des résultats

In [ ]:
df_mapping = pd.DataFrame(mapping_data)

print(f"\n{'='*70}")
print("📊 RÉSULTATS DU MAPPING")
print(f"{'='*70}")

total = len(df_png_list)
print(f"\n✅ Matchés: {matched_count}/{total} ({matched_count/total*100:.1f}%)")
print(f"❌ Non-matchés: {len(unmatched_files)}/{total} ({len(unmatched_files)/total*100:.1f}%)")

if len(df_mapping) > 0:
    print(f"\n📋 Distribution des labels:")
    label_dist = df_mapping['label'].value_counts()
    print(label_dist)
    
    print(f"\n\n📝 Premiers 10 mappings:")
    print(df_mapping.head(10).to_string(index=False))
else:
    print("\n❌ ERREUR: Aucun mapping créé!")

In [ ]:
# Afficher les fichiers non-matchés
if len(unmatched_files) > 0:
    print(f"\n{'='*70}")
    print(f"⚠️  FICHIERS NON-MATCHÉS ({len(unmatched_files)})")
    print(f"{'='*70}")
    
    if len(unmatched_files) <= 20:
        for png_file, reason in unmatched_files:
            print(f"   - {png_file}: {reason}")
    else:
        for png_file, reason in unmatched_files[:10]:
            print(f"   - {png_file}: {reason}")
        print(f"\n   ... et {len(unmatched_files)-10} autres")

## 💾 Section 6: Sauvegarder le mapping

In [ ]:
try:
    df_mapping.to_csv(OUTPUT_CSV, index=False)
    print(f"\n✅ Mapping sauvegardé avec succès!")
    print(f"\n   Fichier: {OUTPUT_CSV}")
    print(f"   Lignes: {len(df_mapping)}")
    print(f"   Colonnes: {list(df_mapping.columns)}")
    print(f"\n   Taille du fichier: {os.path.getsize(OUTPUT_CSV) / 1024:.2f} KB")
except Exception as e:
    print(f"❌ Erreur lors de la sauvegarde: {e}")

## 📈 Section 7: Visualisation et statistiques

In [ ]:
import matplotlib.pyplot as plt

if len(df_mapping) > 0:
    # Graphique de distribution des labels
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogramme
    label_counts = df_mapping['label'].value_counts()
    axes[0].bar(range(len(label_counts)), label_counts.values)
    axes[0].set_xticks(range(len(label_counts)))
    axes[0].set_xticklabels(label_counts.index, rotation=45, ha='right')
    axes[0].set_ylabel('Nombre d\'images')
    axes[0].set_title('Distribution des labels (Train Set)')
    axes[0].grid(axis='y', alpha=0.3)
    
    # Pie chart
    axes[1].pie(label_counts.values, labels=label_counts.index, autopct='%1.1f%%', startangle=90)
    axes[1].set_title('Proportion des labels')
    
    plt.tight_layout()
    plt.savefig(os.path.join(DATASET_FOLDER, 'label_distribution.png'), dpi=100, bbox_inches='tight')
    plt.show()
    
    print(f"✅ Graphique sauvegardé: label_distribution.png")

## ✅ Section 8: Résumé final

In [ ]:
summary = f"""
{'='*70}
✨ MAPPING TERMINÉ AVEC SUCCÈS!
{'='*70}

📊 STATISTIQUES:
   PNG traités: {len(df_png_list)}
   Matchés: {matched_count} ({matched_count/len(df_png_list)*100:.1f}%)
   Non-matchés: {len(unmatched_files)} ({len(unmatched_files)/len(df_png_list)*100:.1f}%)

🏷️  LABELS:
   Nombre de classes: {df_mapping['label'].nunique()}
   Classes: {', '.join(df_mapping['label'].unique())}

📁 FICHIERS GÉNÉRÉS:
   - png_label_mapping.csv (mapping PNG → label)
   - label_distribution.png (visualisation)

{'='*70}
🎯 PRÊT POUR L'ÉTAPE SUIVANTE: Train/Val/Test split + DataLoader
{'='*70}
"""

print(summary)

# Sauvegarder le résumé
with open(os.path.join(DATASET_FOLDER, 'mapping_summary.txt'), 'w') as f:
    f.write(summary)

print("✅ Résumé sauvegardé: mapping_summary.txt")